## Task 1: Load and Inspect the Data

In [1]:
import pandas as pd
import numpy as np

data = {
    'transaction_id': range(1, 21),
    'date': pd.date_range('2024-10-01', periods=20, freq='D'),
    'region': ['North', 'South', 'East', 'West', None, 'North', 'South', None, 'East', 'West',
               'North', 'South', 'East', 'West', 'North', None, 'East', 'West', 'North', 'South'],
    'product_category': ['Electronics', 'Clothing', None, 'Books', 'Electronics', 'Home',
                         'Clothing', 'Books', 'Electronics', None, 'Home', 'Clothing',
                         'Books', 'Electronics', 'Home', 'Clothing', 'Books', 'Electronics',
                         'Home', 'Clothing'],
    'sales_amount': [1200, 450, 890, None, 1500, 670, None, 340, 2100, 780,
                     560, None, 420, 1800, 920, 510, 380, None, 1100, 640],
    'quantity': [2, 5, 3, 1, None, 4, 2, 3, 1, 5,
                 3, 2, None, 1, 4, 3, 2, 1, None, 4],
    'customer_age': [25, 34, None, 45, 29, None, 38, 52, 27, 41,
                     33, None, 48, 26, 35, 42, None, 31, 39, 44],
    'payment_method': ['Credit Card', 'UPI', 'Cash', 'Debit Card', 'Credit Card',
                       'UPI', 'Cash', None, 'Credit Card', 'UPI', 'Debit Card',
                       'Cash', 'Credit Card', None, 'UPI', 'Cash', 'Debit Card',
                       'Credit Card', 'UPI', None]
}

df = pd.DataFrame(data)

# Display basic info
print(df.info())
print('\nMissing values per column:')
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    20 non-null     int64         
 1   date              20 non-null     datetime64[ns]
 2   region            17 non-null     object        
 3   product_category  18 non-null     object        
 4   sales_amount      16 non-null     float64       
 5   quantity          17 non-null     float64       
 6   customer_age      16 non-null     float64       
 7   payment_method    17 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(3)
memory usage: 1.4+ KB
None

Missing values per column:
transaction_id      0
date                0
region              3
product_category    2
sales_amount        4
quantity            3
customer_age        4
payment_method      3
dtype: int64


## Task 2: Handle Missing Values

In [2]:
# 1. Fill region and product_category with mode
df['region'] = df['region'].fillna(df['region'].mode()[0])
df['product_category'] = df['product_category'].fillna(df['product_category'].mode()[0])

# 2. Fill sales_amount with median
df['sales_amount'] = df['sales_amount'].fillna(df['sales_amount'].median())

# 3. Fill quantity with forward fill
df['quantity'] = df['quantity'].ffill()

# 4. Fill customer_age with mean (rounded to integer)
df['customer_age'] = df['customer_age'].fillna(round(df['customer_age'].mean()))

# 5. Drop rows where payment_method is missing
df = df.dropna(subset=['payment_method'])

# Verify no missing values remain
print('Missing values after cleaning:')
print(df.isna().sum())

Missing values after cleaning:
transaction_id      0
date                0
region              0
product_category    0
sales_amount        0
quantity            0
customer_age        0
payment_method      0
dtype: int64


## Task 3: GroupBy Analysis

In [3]:
# 1. Total sales by region
print('Total Sales by Region:')
print(df.groupby('region')['sales_amount'].sum())

# 2. Average sales by product_category
print('\nAverage Sales by Product Category:')
print(df.groupby('product_category')['sales_amount'].mean())

# 3. Group by region and product_category, total sales and quantity
grouped = df.groupby(['region', 'product_category']).agg(
    total_sales=('sales_amount', 'sum'),
    total_quantity=('quantity', 'sum')
).reset_index()
print('\nSales & Quantity by Region and Product Category:')
print(grouped)

# 4. Top 3 region-product combinations by sales
print('\nTop 3 Region-Product Combinations by Sales:')
print(grouped.sort_values('total_sales', ascending=False).head(3))

Total Sales by Region:
region
East     3790.0
North    6460.0
South    1900.0
West     2230.0
Name: sales_amount, dtype: float64

Average Sales by Product Category:
product_category
Books           508.333333
Clothing        680.000000
Electronics    1381.250000
Home            812.500000
Name: sales_amount, dtype: float64

Sales & Quantity by Region and Product Category:
  region product_category  total_sales  total_quantity
0   East            Books        800.0             4.0
1   East         Clothing        890.0             3.0
2   East      Electronics       2100.0             1.0
3  North         Clothing        510.0             3.0
4  North      Electronics       2700.0             3.0
5  North             Home       3250.0            12.0
6  South         Clothing       1900.0             9.0
7   West            Books        725.0             1.0
8   West         Clothing        780.0             5.0
9   West      Electronics        725.0             1.0

Top 3 Region-Produc

## Task 4: Custom Aggregation

In [4]:
# 1. Custom function: sales range
def sales_range(x):
    return x.max() - x.min()

# 2. Sales range per region
print('Sales Range by Region:')
print(df.groupby('region')['sales_amount'].agg(sales_range))

# 3. Multiple metrics using .agg()
print('\nMultiple Metrics by Region:')
multi_agg = df.groupby('region').agg(
    sales_sum=('sales_amount', 'sum'),
    sales_mean=('sales_amount', 'mean'),
    sales_max=('sales_amount', 'max'),
    quantity_sum=('quantity', 'sum'),
    quantity_min=('quantity', 'min')
).reset_index()
print(multi_agg)

Sales Range by Region:
region
East     1720.0
North     990.0
South     275.0
West       55.0
Name: sales_amount, dtype: float64

Multiple Metrics by Region:
  region  sales_sum  sales_mean  sales_max  quantity_sum  quantity_min
0   East     3790.0  947.500000     2100.0           8.0           1.0
1  North     6460.0  922.857143     1500.0          18.0           1.0
2  South     1900.0  633.333333      725.0           9.0           2.0
3   West     2230.0  743.333333      780.0           7.0           1.0
